In [1]:
Output = c("/Users/alexis/Library/CloudStorage/OneDrive-UniversityofNorthCarolinaatChapelHill/CEMALB_DataAnalysisPM/Projects/P1014. Nanoparticles Respiratory Tract/P1014.3. Analyses/P1014.3.1. Group Distribution Analysis/Output")
cur_date = "021026"

library(readxl)
library(openxlsx)
library(tidyverse)
library(reshape2)
library(rlang)
library(limma)

# reading in files
protein_df = data.frame(read_excel("Input/Protein_Data_020926.xlsx", sheet = 2))[,4:13]
protein_df$Dose = as.numeric(protein_df$Dose)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths



Attaching package: ‘rlang’


The following objects are masked from ‘package:purrr’:

    %@%, flatten, flatten_chr, flatten_dbl, flatten_int, flatten_lgl,
    flatten_raw, invoke, splice


Warning message:
“NAs introduced by coercion”


In [2]:
head(protein_df)

,Subject_No,Treatment,Sample_ID,Dose,Separation,ELISA_ID,UniProt_ID,Protein_Name,Conc,Conc_pslog2
,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
1,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03001,Q07011,4-1BB,2.002402,1.586117
2,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03541,Q15109,AGER,3.425807,2.145941
3,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL01701,Q9UNG2,AITRL (GITR Ligand),2.178579,1.668382
4,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL04271,P31749,AKT1,2.214532,1.684609
5,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03101,Q15389,Angiopoietin-1,2.611234,1.852492
6,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL09731,P05089,ARG1,3.369304,2.127404


In [3]:
# only interested in centrifuged samples
protein_df = protein_df %>%
    filter(Separation == 'C')

we are interested in comparing NP1_Rhodamine B and NP2 with Vehicle control. For NP1 and NP1_Rhodamine B, two doses are available. 
Let's answer this with a 

Now on the data analysis side, we are interested in dose-dependent effect of NP1 and NP1_Rhodamine B. A comparison between NP1 and NP1_Rhodamine B will also be informative. Ultimately, the effect of NP2 and NP2_Rhodamine B (with only n=1) at the higher dose will be very informative. Comparison between NP1 and NP2 at the higher dose will be informative too.


## Is there a statistically significant difference in protein expression in respiratory tract cells based on treatment? 

We'll linear modeling to compare, NP1 (at both doses), NP1-Rhodamine B (at both doses) and NP2 to controls.

In [4]:
unique(protein_df$Treatment)
unique(protein_df$Dose)

[1] "Control"         "NP1"             "NP1-Rhodamine B" "NP2_Rhodamine B"
[5] "NP2"

[1]   NA 0.10 0.05

In [5]:
# creating a comparison df to iterate through
comparison_df = data.frame(#Group1 = c(rep(c("Control"), times = 6)),
                           Tx = c(unique(protein_df$Treatment)[2:5], unique(protein_df$Treatment)[2:3]),
                           Dose = c(0.05,0.05,0.1,0.1,0.1,0.1))

comparison_df

Tx,Dose
<chr>,<dbl>
NP1,0.05
NP1-Rhodamine B,0.05
NP2_Rhodamine B,0.10
NP2,0.10
NP1,0.10
NP1-Rhodamine B,0.10


In [10]:
wilcoxon_values = function(df, comp_df){
    # """
    # Running t tests after filtering for set, treatment, exposure, and protein using a loop. 
    # Ultimately using this test to compare different treatment groups to controls.

    # :param: subsetted dataframe, empty dataframe
    # :output: a dataframe containing the set, treatment, exposure, protein, u stat, p value, p adj

    # """
    proteins = unique(df$Protein_Name)
    
    values_df = data.frame()
    # iterating through each tx, protein

    for(i in 1:length(proteins)){
        for(j in 1:length(comp_df$Tx)){
            # unexposed df
            unexposed_df = df %>%
                filter(Protein_Name == proteins[i], Treatment == "Control")
            # exposed df
            exposed_df = df %>%
                filter(Protein_Name == proteins[i], Treatment == comp_df$Tx[j], 
                       Dose == comp_df$Dose[j])

            # wilcoxon test
            wilcox_test = wilcox.test(unexposed_df$Conc_pslog2, exposed_df$Conc_pslog2)
            
            # calculating FC to get directionality
            FC = log2(mean(exposed_df$Conc)/mean(unexposed_df$Conc))

            # contains the treatments compared, dose, protein, test statistic, and p value
            values_vector = cbind("Control", comp_df$Tx[j], comp_df$Dose[j], proteins[i], FC, wilcox_test$statistic, 
                                  wilcox_test$p.value)
            values_df = rbind(values_df, values_vector)
            }
        }

    
    # adding col names
    colnames(values_df) = c("Group1", "Group2", "Dose", "Protein_Name", "log2FC", "Statistic", "P Value")
    
    
   # calculating padj values
    values_df = values_df %>%
        group_by(Group2, Dose) %>%
        mutate(`P Value` = as.numeric(`P Value`),
               log2FC = as.numeric(log2FC),
           `P Adj` =  p.adjust(`P Value`, method = "fdr")) 
    
    return(values_df)
}

# calling fn
control_tx_wilcox_df = wilcoxon_values(protein_df, comparison_df)

head(control_tx_wilcox_df)

Group1,Group2,Dose,Protein_Name,log2FC,Statistic,P Value,P Adj
<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>
Control,NP1,0.05,4-1BB,-0.020094658,8,0.2,0.6409091
Control,NP1-Rhodamine B,0.05,4-1BB,-0.009828542,7,0.4,0.9808696
Control,NP2_Rhodamine B,0.1,4-1BB,-0.041170479,3,0.5,1.0000000
Control,NP2,0.1,4-1BB,-0.003089979,6,0.7,0.8657895
Control,NP1,0.1,4-1BB,0.012048215,3,0.7,0.9444976
Control,NP1-Rhodamine B,0.1,4-1BB,-0.012078402,8,0.2,0.7421053


In [12]:
control_tx_wilcox_df %>%
    filter(Protein_Name == 'TNFRSF11B')#`P Adj` < 0.1)

Group1,Group2,Dose,Protein_Name,log2FC,Statistic,P Value,P Adj
<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>
Control,NP1,0.05,TNFRSF11B,-0.4358076,9,0.1,0.6000000
Control,NP1-Rhodamine B,0.05,TNFRSF11B,-0.1707675,9,0.1,0.8812500
Control,NP2_Rhodamine B,0.1,TNFRSF11B,-0.5372194,3,0.5,1.0000000
Control,NP2,0.1,TNFRSF11B,-0.6235698,9,0.1,0.3481481
Control,NP1,0.1,TNFRSF11B,-0.6635312,9,0.1,0.5423077
Control,NP1-Rhodamine B,0.1,TNFRSF11B,-0.4415180,9,0.1,0.6130435


There are no signficant differences in protein expression between controls and treatment groups regardless of dose.

In [4]:
wide_protein_matrix = protein_df %>%
  select(Sample_ID, UniProt_ID, Conc_pslog2) #%>%
  #pivot_wider(names_from = Sample_ID, values_from = Conc_pslog2) #%>%
  #column_to_rownames("UniProt_ID") %>%
  #as.matrix()

head(wide_protein_matrix)

,Sample_ID,UniProt_ID,Conc_pslog2
,<chr>,<chr>,<dbl>
1,Control_2_NA,Q07011,1.585350
2,Control_2_NA,Q15109,2.135693
3,Control_2_NA,Q9UNG2,1.642756
4,Control_2_NA,P31749,1.670583
5,Control_2_NA,Q15389,1.896098
6,Control_2_NA,P05089,2.120742
